# Lesson 66 · ASR end to end — the full chain from sound wave to text: end-to-end vs the classical pipeline

**Goal**: get a clear view of the whole ASR chain first: features → encoder → decoder → text.

> **A map lesson**: draw the block diagram first, then skim L67–L75; this lesson only grabs the big picture and doesn't dig deep into any one piece (CTC and Whisper get their own lessons later).

🔗 **Aurora connection**: `aurora.audio.mel` implements the feature-extraction layer of ASR (the log-mel spectrogram); `aurora.speech` (new this month) will later hold the CTC / attention encoder–decoders.

← **Previous lesson** · [L65 · Training visualized](../6_deep_learning/L65_visual_training.ipynb)

> Last lesson covered **training visualized**: loss / accuracy curves, gradient norms, weight histograms.
> This lesson explores **ASR end to end**.

## This lesson's story: how does Siri turn what you say into text?

You say a sentence, and within a second Siri has turned it into text. What actually happens in between?

**Why the problem is hard**:
1. Sound is a continuous waveform; text is a sequence of discrete tokens
2. The same word lasts a different time for different speakers ("hello" might take 0.3 s or 0.7 s)
3. The same word sounds wildly different in different contexts (fast speech, accents, noise)

**The three-step chain of end-to-end ASR**:
```
waveform (16kHz) → [acoustic encoder] → feature sequence → [language decoder] → token sequence → text
```

**The key differences between classical and end-to-end**:

| | Classical HMM-GMM | End-to-end (Whisper) |
|---|---|---|
| Acoustic model (AM) | GMM, each frame modeled independently | Transformer encoder, global attention |
| Language model (LM) | Trained separately, errors at the seams | Jointly optimized end to end |
| Decoding | Viterbi, relies on a pronunciation dictionary | Beam search on token logits |
| Training data | Aligned text + audio (expensive) | Any speech + transcript (cheap) |

**Evaluation metric**: WER (word error rate) = the minimum number of word operations needed to turn the hypothesis text into the reference text, divided by the number of reference words:
```
WER = (substitutions S + deletions D + insertions I) / N (number of reference words)
```

This section's task: implement `compute_wer(hypothesis, reference)` — compute WER with Levenshtein dynamic programming (DP).

In [ ]:
import numpy as np

## 1. End-to-end ASR vs classical HMM-GMM

The classical pipeline cuts the task into three pieces that are trained separately: an **acoustic model** (a GMM models frame probabilities) → a **pronunciation dictionary (lexicon)** → a **language model** (N-gram). Errors can't be backpropagated end to end; each piece is tuned to its own best, and the best pieces don't necessarily add up to the best whole.

End-to-end systems (Whisper / wav2vec 2.0) learn P(text | audio) directly: a Transformer encoder processes the frame sequence, and a decoder (or a CTC head) outputs token probabilities. A single run of gradient descent optimizes the whole chain at once, and given enough data it crushes the classical approach.

Architecture comparison (simplified):
```
HMM-GMM: waveform → MFCC → GMM/HMM → lexicon → N-gram LM → text
Whisper:  waveform → log-Mel → Transformer Enc → Transformer Dec → text
```

### Aside: why "encoder" and "decoder"?

The last section said "a Transformer encoder processes the frame sequence, and a decoder outputs token probabilities". Those two names weren't picked at random — they describe exactly what each part does.

**An analogy: shipping a parcel**

- You want to send a fragile vase to a friend. Toss it straight into a box and it will break, so first you **pack** it — wrap it in bubble wrap into a parcel of a standard size (however odd the vase's shape, the parcel's outside is uniform). Packing is like **encoding**: turning something irregular and raw into a tidier, more abstract representation.
- When your friend gets the parcel, they have to peel off the bubble wrap and take out the vase before they can really "see" what's inside. Unpacking is like **decoding**: turning the compressed, abstract representation back into concrete content a person can understand.

**Back in ASR, what actually happens?**

- **Encoder**: the input is the sequence of log-mel frames for a stretch of audio — and different utterances come out at different lengths ("hello" might be 30 frames, or 70). The encoder runs these frames through many Transformer layers and turns every frame into a hidden-state vector of a fixed dimension (say 512). **The key point**: however long the raw audio is, the "representation" for each frame has a fixed size — that is "encoding": compressing / converting the irregular raw acoustic signal into the model's uniform, abstract internal numbers.
- **Decoder**: its input is the hidden states the encoder produced, plus the text it has already written. Step by step, the decoder "translates" those abstract hidden states back into text tokens a human can read — that is "decoding": unfolding the compressed abstract representation, one step at a time, into a concrete, readable result.

**Why not just call them "front half" and "back half"?**
Because the two parts do operations of a completely different nature — one "compresses / abstracts" (turning the concrete sound signal into the model's internal numbers), the other "restores / makes concrete" (turning those numbers back into text people can read). The pair of names "encode / decode" captures that symmetry and direction precisely, while "front / back" only tells you the order and loses that meaning. That's also why nearly every structure of this kind (compress / decompress, encrypt / decrypt, the encoder–decoder architecture) uses this pair of terms.

### ⚠️ Quick concept note: what are an HMM and a GMM?

**This part is written for students meeting these terms for the first time.** If you already know them, feel free to skip it.

Speech recognition talks about "a GMM models frame probabilities" and "HMM state posterior probabilities" — unfamiliar names. What are they?

**GMM (Gaussian mixture model)**:
- Put simply, it represents each **frame of audio** as a weighted sum of a few "bell curves" (Gaussian distributions)
- What is a frame for? It chops the continuous waveform into small windows (10 ms per frame); each frame is treated as a "little snapshot"
- For the phoneme "ah", frame 1 might be "distribution A", frame 2 "distribution B", …
- **The problem**: a GMM only looks at a **single frame** and ignores the links between frames (how "ah" changes over time)

**HMM (hidden Markov model)**:
- Strings the GMM's "frame probabilities" together with a **state machine**
- The state machine's logic: the previous frame is "start of the sound" → the current frame is "middle of the sound" → the next frame is "end of the sound"
- The HMM lets the system remember that a phoneme's pronunciation usually has a **time order**, instead of every frame standing alone
- **Upside**: it adds "time order" on top of the GMM; **downside**: it's complex to implement and needs hand-designed state-transition rules

**Why are these outdated?**
The pain point of the classical approach: the GMM and HMM are trained separately, so "gaps" easily appear when you combine them (improve one component and the whole doesn't improve; another one gets worse along with it). Modern end-to-end systems learn P(text | audio) directly and optimize all parameters at once with one big gradient descent, which avoids the problem.

---

In [ ]:
# Demo: the difference in the "interface" between the two approaches
# The classical approach outputs HMM state posteriors per frame; end-to-end outputs token logits directly

T, n_mels = 300, 80   # 300 frames, 80-dim log-Mel
n_vocab   = 51865     # Whisper vocabulary size

# Classical: each frame → GMM state probabilities (assume 3000 HMM states)
n_hmm_states = 3000
hmm_output_shape = (T, n_hmm_states)

# End-to-end: encoder output sequence → decoder outputs token probabilities
enc_output_shape = (T // 2, 512)   # after 2× downsampling, d_model=512
dec_output_shape = (n_vocab,)       # predicts one token per step

print(f'HMM-GMM output per frame: {hmm_output_shape}')
print(f'Whisper encoder output shape: {enc_output_shape}')
print(f'Whisper decoder output per step: {dec_output_shape}')

### Aside: what are "downsampling" and stride?

The code above mentions "2× downsampling". What does that mean?

**An intuitive picture of downsampling**:
- Imagine a 30 fps video (30 frames a second); now "keep every other frame", and it becomes 15 fps
- Or "keep one frame in every three", and it becomes 10 fps
- And in space? A 1000×1000 high-resolution image, "keep every other pixel", becomes a 500×500 thumbnail

**In a convolutional neural network**:
- `stride=1` means the sliding window moves 1 position at a time, so every position is processed
- `stride=2` means the sliding window moves 2 positions at a time — "skip one, take one"
- The result is an output sequence half as long

**Why does an ASR encoder downsample?**
1. **It saves compute**: 300 frames become 150, so later layers have less to process, and the model runs faster
2. **It keeps the big picture**: tiny timing differences (say 2 frames = 20 ms) usually don't matter for recognizing words; after downsampling, the "large-scale timing patterns" (like a 200 ms rise and fall of intonation) stand out more clearly and are easier for the model to learn

**Whisper's design**: 2 convolution layers, the first with stride=1 and the second with stride=2, for 2× downsampling in total. The factor is an empirical choice (downsample too much and you lose detail; don't downsample at all and it's too slow).

---

### 1b. Input/output shapes and parameter counts of the three main modules

End-to-end ASR breaks down into three reusable modules. Understand each module's interface first, and then you can snap them together into a system:

| Module | Input | Output | Typical parameter count (Whisper-small) |
|---|---|---|---|
| Acoustic encoder (AM / encoder) | log-mel spectrogram | sequence of hidden states | ≈ 88 M |
| Language decoder (LM / decoder) | encoder output + tokens generated so far | next-token logits | ≈ 154 M |
| Beam-search decoder | sequence of logits | list of token ids → text | no trainable parameters |

In classical HMM-GMM the three pieces are **trained separately** and errors can't be backpropagated end to end;
E2E models such as Whisper tie the pieces together with a single cross-entropy loss, which cuts engineering complexity substantially.

In [ ]:
# Demo: the shape changes through AM / LM / Decoder
B, T, n_mels = 2, 300, 80
encoder_d = 512
vocab_size = 51865

# Step 1 — acoustic encoder: T frames → T/2 frames (Whisper uses 2 conv layers; the second has stride=2 → 2× downsampling)
am_out_shape = (B, T // 2, encoder_d)
print(f"AM  output: {am_out_shape}")

# Step 2 — language decoder: generates 1 token per step (autoregressive)
lm_out_shape = (B, vocab_size)
print(f"LM  output (per step): {lm_out_shape}")

# Step 3 — beam search: keep beam_size candidate paths
beam_size = 5
beam_candidates = beam_size
print(f"Beam search: keep {beam_candidates} candidates per step")

# Rough AM / LM parameter counts (Whisper-small, officially 244M parameters in total)
am_params  = 88e6
lm_params  = 154e6
print(f"Whisper-small: AM ≈ {am_params/1e6:.0f}M parameters, LM ≈ {lm_params/1e6:.0f}M parameters")
print(f"Total parameters ≈ {(am_params + lm_params)/1e6:.0f}M")


### Aside: what does "autoregressive" mean? Why does beam search work?

Two new words just showed up in the code — "autoregressive" and "beam search". Both answer the same question: **when the decoder generates tokens step by step, how exactly does "step by step" work?**

**What is autoregressive?**

Take the word apart: `auto` = self, `regress` (used loosely here) = infer / predict. Put together: **autoregressive = inferring the next result from the results you've already produced yourself.**

Think of your phone keyboard's word suggestions: you type "I'm feeling really", and the keyboard uses the words you've already typed to predict that the next word is most likely "happy", "tired" or "busy". It predicts the next step by "looking at what it has already written" — that's autoregression.

Back in the ASR decoder, here's what actually happens:
```
Step 1: only the encoder's hidden states are available (nothing generated yet) → generate token_1
Step 2: encoder hidden states + token_1 (just generated by itself) → generate token_2
Step 3: encoder hidden states + token_1, token_2 (the two it just generated) → generate token_3
...
```
Each step's input includes "all of its own outputs from the previous steps" — that's exactly what the code comment "generates 1 token per step (autoregressive)" means. (The "regression" here is not linear regression from statistics; it's the loose sense of "predict / infer". Don't confuse it with the gradient descent / regression models of L23–25.)

---

**Why does beam search work? How is it different from "always pick the top score"?**

Since generation is autoregressive and step by step, the simplest idea is: at every step pick only "the word that looks most probable right now". That's called **greedy search**. But greedy search has a fatal flaw — think of a maze: if at every step you only take "the turning that looks closest right now", then once you take one wrong turn, everything after is built on that mistake; you can't go back and choose again, and you can easily end up in a dead end (in algorithms this is getting stuck in a **local optimum** rather than the global optimum).

**The idea of beam search**: instead of keeping just 1 path, **keep the `beam_size` (say 5) highest-scoring candidate sequences at the same time** and move them all forward together:
1. At each step, for every candidate sequence currently kept, try every possible next word and compute the new cumulative score
2. Put all the "candidate sequence × every word in the vocabulary" combinations together, keep only the `beam_size` highest-scoring ones, and throw away the rest
3. Repeat until the end-of-sequence token is generated

Because several paths are explored at once, even if one step on some path isn't "the best-looking right now", it still has a chance to survive and win in the end as long as its overall score is higher — that's why beam search finds globally better sequences more easily than greedy search.

**Why is it called a "beam"?** Picture the beam of a flashlight — not a thin line but a spread of light that lights up an area of many paths at once. Beam search "lights up" (explores) several promising candidate paths at once instead of staring at just one.

**How it differs from Viterbi** (following up the "Decoding: Viterbi vs beam search" row of the table): Viterbi is an **exact algorithm** — in a model like an HMM, with a finite number of states and a fixed structure, it is guaranteed to find the globally optimal path. Beam search is an **approximate algorithm** — it doesn't guarantee the global optimum, but its cost stays under control: the "state space" of a Transformer decoder is token sequences of any length, with a vocabulary of tens of thousands of words, so trying every possibility is simply out of the question; beam search trades "keep only a few candidates" for a feasible amount of computation.

In [ ]:
# Demo: greedy search vs beam search — why keeping several candidates finds a better overall result
#
# Setup: we deliberately make greedy search go wrong at the very first step — it picks 'A', which scores higher right now,
# but every path after 'A' is poor; while 'B', which looks lower for the moment, leads on to a high-scoring path.
# scores_table[sequence generated so far] = {next candidate word: its score}
scores_table = {
    (): {'A': 0.6, 'B': 0.4},        # step 1: A leads for now
    ('A',): {'A': 0.1, 'B': 0.1},    # after A, both paths are poor (a dead end)
    ('B',): {'A': 0.9, 'B': 0.1},    # after B, following with A is a great choice
}

def greedy_search(steps=2):
    """Keep only 1 candidate per step: the one with the highest current score."""
    seq, total_score = (), 0.0
    for _ in range(steps):
        next_scores = scores_table[seq]
        best_word = max(next_scores, key=next_scores.get)
        total_score += next_scores[best_word]
        seq = seq + (best_word,)
    return seq, total_score

def beam_search(beam_size=2, steps=2):
    """Keep the beam_size candidate sequences with the highest cumulative score at each step."""
    beams = [((), 0.0)]  # [(sequence so far, cumulative score), ...]
    for _ in range(steps):
        candidates = []
        for seq, score in beams:
            for word, s in scores_table.get(seq, {}).items():
                candidates.append((seq + (word,), score + s))
        candidates.sort(key=lambda x: -x[1])
        beams = candidates[:beam_size]   # keep only the beam_size best
    return max(beams, key=lambda x: x[1])

g_seq, g_score = greedy_search()
b_seq, b_score = beam_search(beam_size=2)

print(f"Greedy search (keeps 1 per step)  result: {g_seq}, total score={g_score:.2f}")
print(f"Beam search (beam_size=2) result: {b_seq}, total score={b_score:.2f}")
print()
print("At step 1 greedy sees A(0.6) > B(0.4) and picks A — but every path after A is poor (0.1).")
print("Beam search keeps both A and B as candidates, and so discovers that B followed by A scores a high 0.9,")
print("finally finding the sequence with the higher total. That's why beam_size > 1 beats greedy (which is beam_size = 1).")

## 2. Features: the 80-dim log-mel spectrogram

Whisper's standard input is an **80-dim log-mel spectrogram**, with exactly the same frame parameters as L43–L47:
```
sample rate  sr     = 16000 Hz
window       n_fft  = 400  samples  (25 ms)
hop          hop    = 160  samples  (10 ms)
mel bins     n_mels = 80
```

The processing chain: `stft → |·|² → mel_filterbank · power → log(max(·, 1e-10))`. The result has shape `(80, T_frames)`, with T_frames ≈ total samples / hop.

Why mel instead of linear frequency? The ear resolves low frequencies more finely, and the mel scale is designed around exactly that property of hearing, which makes it easier for the model to learn phoneme boundaries.

In [ ]:
# Demo: deriving the STFT frame count + the shape of the log-Mel spectrogram

# ===== The STFT frame-count formula: why 1 + (n_samples - n_fft) // hop? =====
#
# The intuition (geometrically):
# Picture a time axis n_samples samples long (say 48000).
# Now slide a "window" (n_fft samples wide) from left to right.
# Each time it slides by hop samples (say 160), compute one FFT.
#
# The key question: starting at position 0, how many times can it slide?
# - window 1: positions 0 to 399 (n_fft-1)
# - window 2: positions 160 to 559
# - window 3: positions 320 to 719
# - ……
# - the last window must satisfy start + n_fft - 1 < n_samples
#   i.e. start < n_samples - n_fft + 1
#   i.e. start ∈ [0, n_samples - n_fft]
#
# If start grows by hop each time, from 0 up to at most n_samples - n_fft:
#   number of windows = floor((n_samples - n_fft) / hop) + 1
#
# That's the formula 1 + (n_samples - n_fft) // hop

sr     = 16000
n_fft  = 400
hop    = 160
n_mels = 80
duration_s = 3.0

n_samples  = int(sr * duration_s)
n_frames   = 1 + (n_samples - n_fft) // hop

print(f'=== Deriving the STFT frame count ===')
print(f'Audio length: {duration_s} s  →  {n_samples} samples')
print(f'Window: n_fft = {n_fft} samples = {n_fft/sr*1000:.1f} ms')
print(f'Hop: hop = {hop} samples = {hop/sr*1000:.1f} ms')
print(f'Formula: n_frames = 1 + ({n_samples} - {n_fft}) // {hop}')
print(f'     = 1 + {n_samples - n_fft} // {hop}')
print(f'     = 1 + {(n_samples - n_fft) // hop}')
print(f'     = {n_frames}')
print(f'\nlog-Mel spectrogram shape: ({n_mels}, {n_frames})')
print(f'Time covered per frame: {n_fft/sr*1000:.1f} ms, spacing between frames: {hop/sr*1000:.1f} ms')

### Going deeper: every step of log-mel feature extraction

Having counted the frames, let's take the chain "stft → |·|² → mel_filterbank · power → log(max(·, 1e-10))" apart step by step.

**Step 1: STFT (short-time Fourier transform)**
- **What it does**: chops the waveform into short windows (25 ms each) and Fourier-transforms each window
- **Input**: a 1-D waveform (a time series)
- **Output**: 2-D data, rows = frequencies, columns = time frames (one spectrum per frame)
- **The values**: complex numbers (each one = the "strength + phase" of one frequency in one frame)

**Step 2: |·|² (squared magnitude)**
- **Purpose**: extract the "energy" from the complex numbers and throw away the phase (phase doesn't help much with recognizing words)
- **How**: if the STFT outputs a complex number `z = a + bi`, then `|z|² = a² + b²`
- **Result**: non-negative real numbers, shape `(number of frequencies, number of frames)`, e.g. `(201, 300)` (n_fft = 400 gives 400/2 + 1 = 201 frequency bins)

**Step 3: mel_filterbank · power (the mel filterbank)**
- **The problem**: the ear's frequency resolution is fine at low frequencies and coarse at high ones (you can hear small pitch differences low down, but not the same-sized differences high up)
- **The fix**: use a "mel filterbank" to squeeze the 201 frequency bins down to 80 (on the mel scale)
  - The mel formula: `mel = 2595 * log10(1 + hz/700)` (from frequency in Hz to the perceptual mel scale)
  - The low end (e.g. 0–1000 Hz) is spread across many filters; the high end (e.g. 4000–8000 Hz) is squeezed into a few
  - That gives the model more "resolving power" for low-frequency detail
- **How it's computed**: a matrix product `(80, 201) @ (201, 300) = (80, 300)`
- **Result**: an 80-dim mel spectrogram (not yet logged)

**Step 4: log(max(·, 1e-10)) (log compression + numerical stability)**
- **Why take the log?**
  - Raw energies span a huge range (from near 0 up to hundreds), which is bad for neural-network learning (the gradients become very unbalanced)
  - After the log, all values are squeezed into a much smaller range; at the same time, differences among small values, which would otherwise be swamped by the big ones, are spread out where the model can tell them apart
  - For example: `log(0.001) ≈ -6.9, log(0.999) ≈ -0.001`, a gap of 6.9; without the log the gap is 0.998, which is easily drowned out next to energies in the hundreds
- **Why max(·, 1e-10)?**
  - Energy can be 0 (silent stretches), but `log(0)` is undefined (it heads to -∞)
  - The max guarantees the input is ≥ 1e-10, so `log(1e-10) ≈ -23`, a finite large negative number
  - The 1e-10 threshold is an empirical choice: small enough (it won't wrongly boost real energies close to 0), and large enough (it avoids floating-point underflow)
- **Result**: an 80-dim log-mel spectrogram, roughly in the range -25 to 10, well suited to a neural network

---

## 3. WER — word error rate

WER (word error rate) is the single most important evaluation metric in ASR:
```
WER = (S + D + I) / N
```
- `N`: the total number of words in the reference (correct) text
- `S` (substitution): number of substitution errors — the model output the wrong word
- `D` (deletion): number of deletion errors — a word was dropped
- `I` (insertion): number of insertion errors — an extra word was added

WER is computed with **edit distance** (Levenshtein distance), counting in words rather than characters. WER can be > 1.0 (when there are lots of insertions).

Common baselines (English):
```
Whisper-large-v3    WER ≈ 2–5%   (LibriSpeech test-clean)
Whisper-base        WER ≈ 5–8%
Classical HMM-GMM   WER ≈ 8–15%
```

In [ ]:
# Demo: tracing one WER calculation by hand
reference  = ['the', 'cat', 'sat', 'on', 'the', 'mat']
hypothesis = ['the', 'cat', 'sat', 'on', 'a',   'mat']
#                                           ^ substitution: the→a

# Count by hand
S = 1   # 'the' → 'a'
D = 0
I = 0
N = len(reference)
wer_manual = (S + D + I) / N

print(f'Reference:  {reference}')
print(f'Hypothesis: {hypothesis}')
print(f'S={S}, D={D}, I={I}, N={N}')
print(f'WER = ({S}+{D}+{I})/{N} = {wer_manual:.4f} = {wer_manual*100:.2f}%')

## 4. ✏️ Implement `compute_wer(hypothesis, reference)`

**Core idea**: word-level Levenshtein edit distance

**DP state**: `dp[i][j]` = the minimum number of operations to turn `hypothesis[:i]` into `reference[:j]`

**Transition**:
```
if hypothesis[i-1] == reference[j-1]:
    dp[i][j] = dp[i-1][j-1]           # same word, no operation needed
else:
    dp[i][j] = 1 + min(
        dp[i-1][j],    # delete hypothesis[i-1]
        dp[i][j-1],    # insert reference[j-1]
        dp[i-1][j-1]   # substitute
    )
```

**Return value**: `edit_distance / len(reference)`

**Acceptance criteria**:

| Input | Expected output |
|---|---|
| `hyp=['the','cat'], ref=['the','cat']` | `0.0` |
| `hyp=['the'], ref=['the','cat']` | `0.5` (1 deletion / 2 words) |
| `hyp=['the','dog'], ref=['the','cat']` | `0.5` (1 substitution / 2 words) |

**Edge case**: when `reference` is empty, return `0.0` if `hypothesis` is also empty; otherwise return `float('inf')` (you can't use 0 words as a reference). This matches the convention of `wer()` in L67.

### Aside: why do deletion, insertion and substitution all cost 1?

A careful student might ask: in the transition above, "substitute", "delete" and "insert" all cost `1`. But intuitively, "weather" misheard as "whether" (just two letters different) feels "very close", while "weather" misheard as a completely unrelated word (say "banana") feels "very far". Why do both cases cost 1 in the formula, treated exactly the same?

**This isn't an oversight; it's the definition of "standard edit distance" itself.**

Standard edit distance (Levenshtein distance) answers a very plain question: **what is the minimum number of basic operations (insert / delete / change) needed to turn one sequence into another?** It treats every operation as an equal "step" — like counting "the fewest intersections between home and the office", where every intersection counts as one step no matter how wide or far apart it is.

**Why simplify like this?**

1. **It avoids subjective arguments**: if substitutions had different costs, you'd first have to answer "how many points should weather → whether lose? And weather → banana?" — by spelling? by pronunciation? by meaning? Different yardsticks give completely different answers, and the arguing never ends. A uniform cost of 1 sidesteps that trap.
2. **It keeps the algorithm clean**: uniform costs make edit distance satisfy the mathematical properties a "distance" should have (such as the triangle inequality); the DP recurrence stays simple and its correctness is easy to prove, which is also why L67 can derive it cleanly from scratch.
3. **It's good enough for the specific job of WER**: ASR evaluation only cares about "how many words were recognized wrongly", not "how badly wrong" — for a voice assistant's downstream task, "weather" misrecognized as "whether" and misrecognized as "banana" are **the same one substitution error**, and both can equally make the command be misunderstood.

**What if you really do need to tell "how badly wrong" apart?**
There is a tool for that, called **weighted edit distance** — give different operations, or different character / word pairs, different costs (for example, make substituting similar-sounding words cheaper). But that's a separate topic beyond standard edit distance; this lesson and L67 only implement the most basic **standard version, with every cost equal to 1**.

In one sentence: a cost of 1 isn't "us choosing to cut corners"; it's the very definition of "minimum number of operations" — to measure something else (like "how badly wrong"), you need a different, weighted ruler.

> **💡 A note on learning order**: this exercise uses a try-first, learn-second approach — try implementing the DP first to build intuition;
> L67 derives the mathematical foundations of the Levenshtein algorithm systematically.
> If you get stuck now, you can skip ahead to L67 and come back to finish this implementation afterwards; either order works.
>
> Edge-case spec (your implementation must handle these):
> - `reference` empty and `hypothesis` also empty → return `0.0` (both sides empty, no errors)
> - `reference` empty but `hypothesis` non-empty → return `float('inf')` (you can't use 0 words as a reference; the error rate counts as infinite)

In [ ]:
def compute_wer(hypothesis: list, reference: list) -> float:
    """Compute the word error rate WER = (S+D+I) / len(reference).

    Parameters:
      hypothesis: list of words output by the model
      reference:  list of words in the correct text

    Returns:
      the WER value (0.0 means perfect, > 1.0 means a great many errors)

    Approach (Levenshtein edit distance):
      Dynamic programming, dp[i][j] = minimum operations to turn hypothesis[:i] into reference[:j]

    Filling the table:
      1. Borders: dp[0][j] = j (insert j words), dp[i][0] = i (delete i words)
      2. Transition: if the words match, dp[i][j] = dp[i-1][j-1];
               otherwise, dp[i][j] = 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])
      3. Return: dp[H][R] / R, where H, R = len(hypothesis), len(reference)

    Optimization hint: use a rolling array that keeps only one row of dp values, O(R) space
    """
    H, R = len(hypothesis), len(reference)

    # ✏️ TODO: edge cases (when R==0: return 0.0 if H==0, else float('inf'))

    # ✏️ TODO: initialize the dp array (length R+1: the cost of turning the empty sequence into ref[:j])

    # ✏️ TODO: loop over each word of hypothesis, updating dp (rolling array, O(R) space)

    # ✏️ TODO: return dp[R] / R
    raise NotImplementedError(
        "Please implement compute_wer: fill in all the TODOs following the pseudocode above, then run this cell."
        " For full guidance, you can finish L67 first and come back."
    )

### Helper: understanding the Levenshtein DP, with pseudocode

The formula and transition above may look abstract. Let's use a concrete example to see who "delete", "insert" and "substitute" actually act on, and how to fill in the DP table.

**What the DP table means**:

`dp[i][j]` = the **minimum number of operations** needed to turn `hypothesis[:i]` (the first i words of the hypothesis) into `reference[:j]` (the first j words of the reference).

**A worked example: aligning hypothesis and reference text**

```
hypothesis = ['the', 'dog', 'sat']
reference  = ['the', 'cat', 'sat']
```

We fill a 4×4 DP table (including the empty prefixes, so (H+1) × (R+1) = 4 × 4):

```
        ''   'the'  'cat'  'sat'
   ''    0     1      2     3        ← from empty to the first j reference words: j insertions
'the'    1     0      1     2
'dog'    2     1      1     2
'sat'    3     2      2     1

   ↑     ↑
   from  these rows: from the first i hypothesis words to empty, i deletions
   empty
   to hyp
```

**Filling cell by cell**:

1. **Initialize the first row and the first column**:
   - First row `dp[0][j]`: going from the empty sequence to ref[:j] takes j **insertions** (adding the j words one by one)
   - First column `dp[i][0]`: going from hyp[:i] to the empty sequence takes i **deletions** (removing the i words one by one)

2. **Fill dp[1][1]** (turn "the" into "the"):
   - `hypothesis[0] == reference[0]` (both "the"), so no operation is needed
   - `dp[1][1] = dp[0][0] = 0`  ✅

3. **Fill dp[1][2]** (turn "the" into ["the", "cat"]):
   - `hypothesis[0] = 'the'` differs from `reference[1] = 'cat'`
   - Pick the cheapest way:
     - from dp[0][2]=2 (turning empty into ["the","cat"] takes 2 insertions) + 1 operation (delete hyp[0]) = 3
     - from dp[1][1]=0 (turning "the" into "the" takes 0 operations) + 1 operation (insert "cat") = 1  ← **minimum**
     - from dp[0][1]=1 (turning empty into "the" takes 1 insertion) + 1 operation (substitute hyp[0] with "cat") = 2
   - `dp[1][2] = 1 + min(2, 0, 1) = 1 + 0 = 1`

4. **Fill dp[2][2]** (turn ["the","dog"] into ["the","cat"]):
   - `hypothesis[1] = 'dog'` differs from `reference[1] = 'cat'`
   - Pick one of three:
     - dp[1][2]=1 + 1 (delete hyp[1]) = 2
     - dp[2][1]=1 + 1 (insert ref[1]) = 2
     - dp[1][1]=0 + 1 (substitute hyp[1] with ref[1]) = 1  ← **minimum**
   - `dp[2][2] = 1`

...and so on, up to `dp[3][3] = 1` (of the three words, only the middle one differs).

**Final WER = dp[3][3] / 3 = 1/3 ≈ 0.333**

---

**Pseudocode (for your implementation)**:

```python
def compute_wer(hypothesis, reference):
    H, R = len(hypothesis), len(reference)

    # edge cases
    if R == 0:
        return 0.0 if H == 0 else float('inf')

    # initialize: we only need to keep one row (rolling-array optimization)
    prev_row = list(range(R + 1))  # [0, 1, 2, ..., R]

    # loop over each word of the hypothesis
    for i in range(1, H + 1):
        curr_row = [i]  # first cell: going from hyp[:i] to empty takes i deletions

        # loop over each word of the reference
        for j in range(1, R + 1):
            if hypothesis[i - 1] == reference[j - 1]:
                # same word: inherit directly
                curr_row.append(prev_row[j - 1])
            else:
                # different: take the minimum of the three operations
                delete_cost = prev_row[j] + 1       # delete hypothesis[i-1]
                insert_cost = curr_row[j - 1] + 1   # insert reference[j-1]
                replace_cost = prev_row[j - 1] + 1  # substitute
                curr_row.append(min(delete_cost, insert_cost, replace_cost))

        prev_row = curr_row  # the current row becomes the previous row for the next pass

    # the final answer is in prev_row[-1] (i.e. dp[H][R])
    return prev_row[-1] / R
```

Key points:
- **Rolling array**: keep only the previous row of dp values and replace it once the current row is computed, cutting space from O(H×R) to O(R)
- **What the operations mean**:
  - Delete: `prev_row[j]` means "the hypothesis has been processed up to word i-1 and the reference up to word j"; now delete the hypothesis's word i without touching the reference
  - Insert: `curr_row[j-1]` means "the hypothesis has been processed up to word i and the reference up to word j-1"; now insert the reference's word j into the hypothesis
  - Substitute: `prev_row[j-1]` means "the hypothesis has been processed up to word i-1 and the reference up to word j-1"; now replace the hypothesis's word i with the reference's word j

---

In [ ]:
try:
    # Check compute_wer
    assert abs(compute_wer(['the', 'cat'], ['the', 'cat']) - 0.0) < 0.001, 'identical text should give WER 0'
    assert abs(compute_wer([], ['a']) - 1.0) < 0.001, 'everything deleted should give WER 1'
    assert abs(compute_wer(['the', 'cat', 'sat'], ['the', 'sat', 'cat']) - 2/3) < 0.001, \
        'swapped order (2 substitutions) should give WER 2/3'
    assert abs(compute_wer(['a', 'b', 'c', 'd'], ['b', 'c']) - 1.0) < 0.001, \
        'extra words cause insertion errors'
    print('✅ compute_wer passes all checks')
except (NotImplementedError, TypeError) as _e:
    print(f"⚠️  compute_wer not implemented yet — finish the TODOs and rerun: {_e}")

## 5. Parameter experiment: analyzing error types

Simulate ASR transcripts of 10 sentences, tally the S/D/I counts, and watch how much each error type contributes to WER.

**What to expect**:
- Background-noise scenario: a high share of **D (deletions)** — weakly spoken syllables get swallowed
- Accent scenario: a high share of **S (substitutions)** — phonemes get confused
- Very fast speech: **D** rises again — word boundaries get lost

In the experiment, edit the `pairs` list, swap in your own transcripts, and watch the statistics change.

In [ ]:
# Simulated reference text and ASR hypotheses for 10 sentences (typed in by hand or from real transcripts)
pairs = [
    # (hypothesis_words, reference_words)
    (['hello', 'world'],                    ['hello', 'world']),           # perfect
    (['the', 'cat', 'sat'],                 ['the', 'cat', 'sat', 'down']),# deletion
    (['i', 'love', 'music'],                ['i', 'love', 'music']),       # perfect
    (['this', 'is', 'a', 'test'],           ['this', 'is', 'test']),       # insertion
    (['speech', 'recognition'],             ['speech', 'recognition']),    # perfect
    (['the', 'whether', 'is', 'nice'],      ['the', 'weather', 'is', 'nice']),# substitution
    (['aurora', 'audio'],                   ['aurora', 'audio', 'ai']),    # deletion
    (['deep', 'learning', 'rocks'],         ['deep', 'learning', 'rocks']),# perfect
    (['transformer', 'model'],              ['transformer', 'models']),    # substitution
    (['end', 'to', 'end', 'system'],        ['end', 'to', 'end', 'system']),# perfect
]

try:
    wers = [compute_wer(h, r) for h, r in pairs]
    avg_wer = np.mean(wers)

    print('Per-sentence WER:')
    for i, (w, (h, r)) in enumerate(zip(wers, pairs), 1):
        status = '✅' if w == 0 else '❌'
        print(f'  sentence {i:2d} {status}  WER={w:.3f}  ref_len={len(r)}')
    print(f'\nAverage WER = {avg_wer:.4f} = {avg_wer*100:.2f}%')
except (NotImplementedError, TypeError):
    print('⚠️  compute_wer not implemented yet — finish the ✏️ exercise above before running this parameter experiment')

## Wrap-up

This section implemented `compute_wer`, boiling word-level edit distance down to the WER `(S+D+I)/N`; later training and evaluation will reuse it. The next lesson (L67) implements the Levenshtein dynamic program from scratch, laying the mathematical foundation for WER.

`compute_wer` relies on the edit-distance DP of the next lesson; if you want to do the WER exercise first, it's a good idea to read L67 and then come back.

## ✏️ Whiteboard challenge: derive the core ASR concepts by hand (target: 10 minutes)

Cover the screen and work it out on paper:

**Question 1**: What is Whisper's standard input feature? What are the frame parameters?
(80-dim log-mel, n_fft=400, hop=160, sr=16000 → one frame every 10 ms, 1 second ≈ 100 frames)

**Question 2**: What is `compute_wer(['the', 'dog', 'sat'], ['the', 'cat', 'sat'])`?
(1 substitution / 3 words = 1/3 ≈ 0.333)

**Question 3**: Can WER exceed 1.0? When?
(Yes; when the hypothesis has far more words than the reference, the insertions pile up and WER = (S+D+I)/N can be > 1)

**Question 4**: Why does end-to-end ASR handle a new language more easily than classical HMM-GMM?
(No hand-designed pronunciation dictionary is needed; you only need audio–text pairs, and you can train jointly on weakly supervised multilingual data)

**Question 5**: The acoustic encoder outputs a `(B, T, 512)` tensor. How does the decoder know where to "attend"?
(Cross-attention: decoder queries × encoder keys/values learn the time alignment automatically, with no hand-made forced alignment)

When you've finished, run the cell below to check.

### Aside: an intuitive view of cross-attention

Question 5 of the whiteboard challenge involves "cross-attention", the core mechanism of a Transformer decoder. If you haven't studied Transformers yet, here's a quick explanation.

**The setting**:
- The encoder output has shape `(B, T_enc, 512)`: B audio clips, each with T_enc frames (e.g. 3000), each frame a 512-dim hidden state
- The decoder needs to generate T_dec words (e.g. 20), outputting one token per step
- The question: at each step, which encoder frames should the decoder "attend to"?

**Self-attention vs cross-attention**:

**Self-attention** (the decoder attending to itself):
- The decoder looks only at the tokens it has generated so far (the first k)
- Purpose: understand "the meaning so far" (for example, having generated "the dog", when predicting the third word it takes into account that "the subject is singular" and leans towards the singular form of the verb)

**Cross-attention** (decoder × encoder):
- The decoder looks at **all** the encoder's audio frames
- Purpose: align automatically — "which word am I generating now, and which part of the audio should I look at?"
- No hand-labeled word–frame alignment needed; the model learns it itself

**How cross-attention is computed**:

```
Q (Query)   = decoder hidden state at the current step   shape (B, T_dec, d_model)
K (Key)     = encoder hidden states                      shape (B, T_enc, d_model)
V (Value)   = encoder hidden states                      shape (B, T_enc, d_model)

Scores = Q @ K.T / sqrt(d_model)       shape (B, T_dec, T_enc)
         ↑
         a "similarity matrix": each row is a decoding step, each column an encoder frame

Attention = Softmax(Scores, dim=-1)    shape (B, T_dec, T_enc)
            (normalizes each row along the encoder-frame axis)
         ↑
         so for decoding step t, every encoder frame s gets a weight

Output = Attention @ V                 shape (B, T_dec, d_model)
         (a weighted average over encoder frames)
```

**The intuition**:
- **Q·K.T** measures the similarity between "the word being decoded now" and "some encoder frame"
- **Softmax** turns the similarities into normalized weights (summing to 1)
- **weights × V** takes a weighted average of the encoder frames' hidden states

For example, when the decoder is generating "cat", it might give large weights to encoder frames 500–700 (the stretch of audio where "cat" is spoken) and small weights to the others.

**Why divide by sqrt(d_model)?**
- When d_model is large, the values of Q·K get large, which makes the softmax gradients tiny (vanishing gradients)
- Dividing by sqrt(d_model) keeps the variance of the scores around 1 and stabilizes training (this is called **scaled dot-product attention**)

---

In [ ]:
# ✏️ Answer-check cell
import numpy as np

# Q1: Whisper feature shape (no real audio needed)
sr, n_fft, hop, n_mels = 16000, 400, 160, 80
duration_s = 30.0  # Whisper uses a fixed 30-second window
# Whisper pads/centers before framing, so frames = samples / hop (the no-padding formula 1+(N-n_fft)//hop would give 2998)
n_frames_30s = int(sr * duration_s) // hop
assert n_frames_30s == 3000, f"frames for 30 s={n_frames_30s}"
print(f"Q1 ✅  Whisper input: 80-dim log-Mel, 30 s → {n_frames_30s} frames (hop=10ms)")

# Q2: check the WER calculation
def _ref_wer(hyp, ref):
    if not ref: return 0.0
    H, R = len(hyp), len(ref)
    dp = [[0]*(R+1) for _ in range(H+1)]
    for i in range(H+1): dp[i][0] = i
    for j in range(R+1): dp[0][j] = j
    for i in range(1, H+1):
        for j in range(1, R+1):
            if hyp[i-1] == ref[j-1]: dp[i][j] = dp[i-1][j-1]
            else: dp[i][j] = 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])
    return dp[H][R] / R

wer_q = _ref_wer(['the','dog','sat'], ['the','cat','sat'])
assert abs(wer_q - 1/3) < 1e-9, f"wer={wer_q}"
print(f"Q2 ✅  WER(['the','dog','sat'], ['the','cat','sat']) = {wer_q:.4f} = 1/3")

# Q3: check that WER can exceed 1.0
wer_over1 = _ref_wer(['a','b','c','d','e'], ['x'])   # 4 insertions + 1 substitution = 5, /1 = 5.0
assert wer_over1 > 1.0
print(f"Q3 ✅  WER can exceed 1.0: hyp=['a','b','c','d','e'], ref=['x'] → WER={wer_over1:.1f}")

# Q4: end-to-end WER sanity check (run a few pairs through the reference implementation)
test_pairs = [
    (['hello', 'world'], ['hello', 'world'], 0.0),
    (['hello'],          ['hello', 'world'], 0.5),
    (['hi', 'world'],   ['hello', 'world'], 0.5),
]
for hyp, ref, expected in test_pairs:
    got = _ref_wer(hyp, ref)
    assert abs(got - expected) < 1e-9, f"hyp={hyp},ref={ref}: {got}≠{expected}"
print(f"Q4 ✅  End-to-end ASR evaluation: 3 WER calculations check out (0.0/0.5/0.5)")

# Q5: check the cross-attention shapes
#
# The cross-attention workflow:
#   1. hidden state of the decoder's current word Q: (B, 1, d_model) ← only 1 position per step (the current output position)
#   2. hidden states of all encoder frames K, V: (B, T_enc, d_model) ← every audio frame is available
#   3. compute similarities: Q @ K.T: (B, 1, T_enc) ← "the current word's attention weight on each encoder frame"
#   4. softmax and weighted average:
#      attention_weights = Softmax(scores / sqrt(d_model), dim=-1)  [B, 1, T_enc]
#      context = attention_weights @ V  [B, 1, d_model]
#
# In practice the decoder generates T_dec words, so this computation happens T_dec times;
# stacked together they make the full (B, T_dec, T_enc) attention scores.

B, T_enc, T_dec, d_model = 2, 3000, 20, 512
print(f"\n→ Q5 Cross-attention in detail:")
print(f"  encoder output (B, T_enc, d_model) = ({B}, {T_enc}, {d_model})")
print(f"  decoder output per step (B, 1, d_model) = ({B}, 1, {d_model})")
print(f"  Cross-attention scores: ({B}, {T_dec}, {T_enc})")
print(f"    ↑ for every decoding step, we compute its attention over all {T_enc} encoder frames")

# Demo: single-step cross-attention
Q_single = np.random.randn(B, 1, d_model)
K = np.random.randn(B, T_enc, d_model)
V = np.random.randn(B, T_enc, d_model)
scores_single = (Q_single @ K.transpose(0, 2, 1)) / np.sqrt(d_model)
assert scores_single.shape == (B, 1, T_enc), f"wrong single-step shape {scores_single.shape}"
print(f"  ✓ single-step scores shape: {scores_single.shape}")

# Demo: full cross-attention (T_dec steps)
Q_full = np.random.randn(B, T_dec, d_model)
scores_full = (Q_full @ K.transpose(0, 2, 1)) / np.sqrt(d_model)
assert scores_full.shape == (B, T_dec, T_enc), f"wrong full shape {scores_full.shape}"
print(f"  ✓ full scores shape: {scores_full.shape}")

# Softmax and the values
attention = scores_full  # softmax normalizes along the T_enc axis (axis -1)
attention_weights = np.ones((B, T_dec, T_enc)) / T_enc  # simplified (it should really be a softmax)
context = attention_weights @ V
assert context.shape == (B, T_dec, d_model)
print(f"  ✓ after attention weighting: {context.shape}")

print(f"\n🎉 Cross-attention passed! Through this mechanism the decoder learns on its own to align audio with text")
print("   End of lesson. L67 implements the Levenshtein DP from scratch and builds the maths behind WER.")

In [ ]:
# ✏️ Self-review for this lesson
l66_review = {
    "e2e_vs_hmm_understood":   None,  # understand the key differences between end-to-end and classical HMM-GMM? True/False
    "wer_formula_memorized":   None,  # remember WER=(S+D+I)/N and can compute simple examples by hand? True/False
    "compute_wer_impl":        None,  # compute_wer() DP implemented correctly, all 4 assertions pass? True/False
    "whisper_features":        None,  # remember Whisper input = 80-dim log-Mel, 30 s → 3000 frames? True/False
    "whiteboard_passed":       None,  # all 5 whiteboard questions done? True/False
}

unfilled = [k for k, v in l66_review.items() if v is None]
assert not unfilled, f'not filled in yet: {unfilled}'
weak = [k for k, v in l66_review.items() if v is False]
if weak:
    print(f'⚠️  needs more work: {weak}')
else:
    print('✅ L66 fully cleared! On to L67: edit distance from scratch')

---

→ **Next lesson** · [L67 · Edit distance from scratch](L67_edit_distance.ipynb)

> Next lesson covers **edit distance from scratch**: Levenshtein dynamic programming, the maths of WER.